# 🧪 Helmet Detection: Model Comparison Experiments (Kaggle GPU)

Trains every config in `configs/experiments/` with the same recipe and saves the results as this notebook's **Output**.

| Experiment | What it tests |
|---|---|
| `yolo26s_640` | YOLO26 small (NMS-free, CPU/edge-optimised) |
| `baseline_yolov8s_640` | Original recipe (reference) |
| `yolo26s_640_oversample` | Class imbalance fix: violation images repeated 2× |
| `yolo26n_640` | YOLO26 nano (fastest) |
| `yolov8s_416` | Native image resolution (416) vs upscaled 640 |
| `yolo11s_640` | YOLO11 small |
| `yolov8n_640` | Smaller/faster v8 |

## Before you start
1. **Push the latest code to GitHub.** This notebook clones the repo.
2. **Settings (right panel):**
   - **Accelerator:** GPU P100 or GPU T4 ×2
   - **Internet:** On (needs a phone-verified Kaggle account)
3. **Add-ons → Secrets:** add `ROBOFLOW_API_KEY`, and tick it for this notebook.

## How to run
Use **Save Version → Save & Run All (Commit)**. It runs in the background for up to 12 hours, even if you close the browser, and keeps everything in `/kaggle/working` as the notebook's Output. All 7 experiments take about 5–8 hours.

**Resuming:** if a run stops before finishing, open the notebook, click **+ Add Input → Your Work → this notebook's output**, and commit again. Experiments that already finished are copied over and skipped.

In [ ]:
# Stop early if no GPU is attached (otherwise training would silently run on CPU for hours)
import shutil, subprocess
if shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "No GPU found. In the notebook editor: Settings → Accelerator → GPU P100 (or GPU T4 x2), "
        "wait until the session restarts with the GPU, then Save Version → Save & Run All again."
    )
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

## 1. Setup: code and packages

In [ ]:
import os, shutil, glob

REPO_URL = "https://github.com/Md-Faiz-Alam/helmet_detection.git"
CODE_DIR = "/tmp/helmet_detection"   # code + dataset (not saved as output)
OUT_DIR = "/kaggle/working"           # saved as the notebook's Output
DEVICE = "0"                          # one GPU; "0,1" would use both T4s (multi-GPU training)

!rm -rf {CODE_DIR}
!git clone -q {REPO_URL} {CODE_DIR}
%cd {CODE_DIR}

# Kaggle already has a CUDA build of PyTorch; install only what training needs.
%pip install -q ultralytics==8.4.216 mlflow roboflow

## 2. Dataset

In [ ]:
from kaggle_secrets import UserSecretsClient  # type: ignore  (only exists on Kaggle)
from roboflow import Roboflow  # type: ignore  (installed by %pip above)

if not os.path.isdir("data/train"):
    try:
        api_key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
    except Exception as e:
        raise RuntimeError(
            "Could not read the Kaggle secret ROBOFLOW_API_KEY. In the notebook editor: Add-ons → Secrets → "
            "add a secret with label ROBOFLOW_API_KEY (your Roboflow *private* key) and tick its checkbox "
            "so it is attached to this notebook. Also check Settings → Internet is On."
        ) from e
    rf = Roboflow(api_key=api_key)
    rf.workspace("bike-helmets").project("bike-helmet-detection-2vdjo").version(2).download("yolov8", location="/tmp/rf_download")
    for split in ["train", "valid", "test"]:
        shutil.move(f"/tmp/rf_download/{split}", f"data/{split}")

!python scripts/make_oversampled_split.py --repeat 2

## 3. Output folders, resume, and experiment tracking (MLflow)

- Training writes to `results/train/`, which is linked to `/kaggle/working/train`, so every run is saved as Output.
- Finished runs from an attached previous output (**+ Add Input**) are copied in and skipped.
- Ultralytics logs parameters and per-epoch metrics to MLflow in `/kaggle/working/mlflow.db`. To browse them on your PC, download that file and run `mlflow ui --backend-store-uri sqlite:///mlflow.db`.

In [ ]:
TRAIN_OUT = f"{OUT_DIR}/train"
os.makedirs(TRAIN_OUT, exist_ok=True)
os.makedirs("results", exist_ok=True)
if not os.path.islink("results/train"):
    shutil.rmtree("results/train", ignore_errors=True)
    os.symlink(TRAIN_OUT, "results/train")

# Resume: copy finished runs from a previous version's output, if one is attached as input
for best in glob.glob("/kaggle/input/*/train/*/weights/best.pt"):
    run_dir = os.path.dirname(os.path.dirname(best))
    name = os.path.basename(run_dir)
    if not os.path.exists(f"{TRAIN_OUT}/{name}/weights/best.pt"):
        shutil.copytree(run_dir, f"{TRAIN_OUT}/{name}", dirs_exist_ok=True)
        print(f"♻️  Restored finished run: {name}")

# MLflow now needs a database backend (the plain-folder store is deprecated)
os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{OUT_DIR}/mlflow.db"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "helmet-detection"
# Log only to MLflow; Kaggle preinstalls other trackers that Ultralytics would also try to use
!yolo settings mlflow=True clearml=False comet=False dvc=False wandb=False tensorboard=False raytune=False

## 4. Train all experiments

Comment out lines in `EXPERIMENTS` to skip runs. Order = priority.

In [ ]:
EXPERIMENTS = [
    "yolo26s_640",
    "baseline_yolov8s_640",
    "yolo26s_640_oversample",
    "yolo26n_640",
    "yolov8s_416",
    "yolo11s_640",
    "yolov8n_640",
]

for name in EXPERIMENTS:
    if os.path.exists(f"results/train/{name}/weights/best.pt"):
        print(f"⏭️  {name}: already done")
        continue
    shutil.rmtree(f"results/train/{name}", ignore_errors=True)   # remove a half-finished run
    print(f"🚀 Training {name}")
    !python scripts/train.py --config configs/experiments/{name}.yaml --device {DEVICE}

## 5. Quick comparison (GPU)

The accuracy columns are final. The CPU latency column is measured on Kaggle's CPU, so for a fair speed comparison re-run the benchmark on your own PC (step 6).

In [ ]:
!python scripts/benchmark.py --device {DEVICE} --report {OUT_DIR}/model_comparison_kaggle.md

## 6. Download results for your PC

Creates `/kaggle/working/experiments.zip` with the best weights and training curves of every run. Download it from the notebook's **Output** tab, extract it into `results/train/` in the project, and run:

```
python scripts/benchmark.py
```

In [ ]:
import zipfile
zip_path = f"{OUT_DIR}/experiments.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for run in sorted(glob.glob(f"{TRAIN_OUT}/*/")):
        name = os.path.basename(os.path.normpath(run))
        for f in ["weights/best.pt", "results.csv", "results.png", "experiment.yaml", "args.yaml",
                  "confusion_matrix_normalized.png", "BoxPR_curve.png"]:
            p = os.path.join(run, f)
            if os.path.exists(p):
                z.write(p, f"{name}/{f}")
print("Saved", zip_path, f"({os.path.getsize(zip_path)/1e6:.0f} MB)")